# Feature Tokenizer Transformer

- FT-Transformer with continuous and discretised feature tokenisation
- Benchmark models: XGBoost, LightGBM, Fama-French five-factor Ridge classifier, MLP
- Rolling expanding-window evaluation with per-window no-lookahead preprocessing
- Interpretability: Attention Rollout, Integrated Gradients, GradientSHAP, TreeSHAP
- Attribution temporal stability analysis and Fama-French factor alignment

### packages

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


from typing import Optional, List

import numpy as np

## Transformer

In [ ]:
class ScalarTokeniser(nn.Module):
    # Feature k gets its own weight vector W_k of size d_model
    # Token_k = x_k * W_k + b_k
    def __init__(self, n_features: int, d_model: int):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_features, d_model))
        self.b = nn.Parameter(torch.zeros(n_features, d_model))
        nn.init.kaiming_uniform_(self.W, a=np.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, F]  ->  [B, F, D]
        return x.unsqueeze(-1) * self.W.unsqueeze(0) + self.b.unsqueeze(0)

#TODO: Add Categorical Tokenizer

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_h = d_model // n_heads
        self.scale = self.d_h ** -0.5

        self.qkv = nn.Linear(d_model, 3 * d_model, bias=True)
        self.proj = nn.Linear(d_model, d_model, bias=True)
        self.drop = nn.Dropout(dropout)

        # Attention weights are cached after every forward pass.
        # The rollout method reads this buffer without re-running the model.
        self._cached_weights: Optional[torch.Tensor] = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, S, D = x.shape
        qkv = self.qkv(x).view(B, S, 3, self.n_heads, self.d_h).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)

        scores = (q @ k.transpose(-2, -1)) * self.scale
        weights = scores.softmax(dim=-1)
        self._cached_weights = weights.detach().clone()

        out = (self.drop(weights) @ v).transpose(1, 2).contiguous().view(B, S, D)
        return self.proj(out)

    @property
    def attention_weights(self) -> Optional[torch.Tensor]:
        return self._cached_weights


class ReGLU(nn.Module):
    # Rectified Gated Linear Unit. Splits the channel dimension in half
    # and uses the second half as a ReLU gate for the first half.
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        a, b = x.chunk(2, dim=-1)
        return a * F.relu(b)


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff * 2)   # x2 because ReGLU halves channels
        self.act = ReGLU()
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.drop(self.act(self.fc1(x))))


class EncoderBlock(nn.Module):
    # Pre-LN layout: normalise before attention and FFN, not after.
    # Empirically more stable than post-LN for small tabular Transformers
    # where token sequence length is short.
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.ffn(self.ln2(x)))
        return x

In [ ]:
class FT_Transformer(nn.Module):
    def __init__(self, n_features: int, d_model: int, n_heads: int,
                 n_layers: int, d_ff: int, n_classes: int,
                 dropout: float = 0.1, mode: str = "scaler"):
        #TODO: Need to add bins argument for continuous tokenization
        super().__init__()
        self.mode = mode
        self.n_features = n_features

        self.tokeniser = ScalarTokeniser(n_features, d_model)
        #TODO: Needed to add condition for Continuous Tokenizer

        self.cls_token = nn.Parameter(torch.empty(1, 1, d_model))
        nn.init.normal_(self.cls_token, std=0.02)

        self.blocks = nn.ModuleList([
            EncoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])

        self.final_ln = nn.LayerNorm(d_model)

        self.head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, n_classes)
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def tokenise(self, x: torch.Tensor) -> torch.Tensor:
        tokens = self.tokeniser(x)    # [B, F, D]
        cls = self.cls_token.expand(x.size(0), -1, -1)  # [B, 1, D]
        return torch.cat([cls, tokens], dim=1)     # [B, F+1, D]

    def encode(self, tokens: torch.Tensor) -> torch.Tensor:
        for block in self.blocks:
            tokens = block(tokens)
        return self.final_ln(tokens)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        tokens = self.tokenise(x)
        encoded = self.encode(tokens)
        return self.head(encoded[:, 0])   # classification from CLS token

    def get_attention_weights(self) -> List[Optional[torch.Tensor]]:
        return [blk.attn.attention_weights for blk in self.blocks]


class MLPBaseline(nn.Module):
    # Depth and parameter matched feedforward network.
    # Performance differences relative to FT-Transformer isolate
    # the specific contribution of the inter-feature attention mechanism.
    def __init__(self, n_features: int, d_model: int, n_layers: int,
                 n_classes: int, dropout: float = 0.1):
        super().__init__()
        layers: List[nn.Module] = []
        in_d = n_features
        for _ in range(n_layers):
            layers += [nn.Linear(in_d, d_model), nn.LayerNorm(d_model),
                       nn.ReLU(), nn.Dropout(dropout)]
            in_d = d_model
        layers += [nn.Linear(d_model, d_model // 2), nn.ReLU(),
                   nn.Dropout(dropout), nn.Linear(d_model // 2, n_classes)]
        self.net = nn.Sequential(*layers)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def count_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters() if p.requires_grad)